## 4.3 RNN网络结构 - 输入与输出模式

#### 1. 这一节我们要解决什么问题 🎯

在前面的小节中，我们已经理解了两件事：

* 第一，RNN层本身最核心的任务是递推计算隐藏状态。  
  也就是：  
  当前输入 + 上一时刻隐藏状态 $\rightarrow$ 当前隐藏状态
* 第二，隐藏状态本身并不等于最终任务输出。

通常还需要再经过后续的全连接层，才能映射成真正的输出结果。

那么接下来就会自然出现一个问题：

> 既然 RNN 会在很多个时间步上不断产生隐藏状态，那么最终输出到底应该从哪一步来？

也就是说：

* 是每一步都输出？
* 还是只在最后一步输出？
* 输入和输出的时间步数一定要一样吗？
* 不同任务下，完整模型（RNN层 + FC层）是应该怎样组合？

#### 2. 先区分清楚：隐藏状态 和 最终输出 不是一回事 🧠

##### 2.1 RNN层先产生的是隐藏状态

RNN层在每一个时间步最核心的计算结果是：

$h_t$

它表示：

> 模型在第 $t$ 个时间步，对到当前为止历史信息的一种内部表示。

它更像是：

* 内部特征
* 上下文记忆
* 时序信息的压缩结果

所以隐藏状态本质上是 RNN层内部的特征表示，而不是最终任务答案。

##### 2.2 最终输出通常是在隐藏状态基础上再映射得到的

如果要得到真正的任务输出，通常还要再接一个输出层，例如全连接层：

$o_t = W_y h_t + b_y$

如果任务还需要概率形式，再接激活函数：

$y_t = g(o_t)$

所以更完整的单步结构应该理解成：

$x_t + h_{t-1} \rightarrow h_t \rightarrow o_t \rightarrow y_t$

这里：

* $h_t$：RNN层输出的隐藏状态
* $o_t$：输出层的线性结果
* $y_t$：最终任务输出

##### 2.3 所以这一节讨论的“输出”，指的是最终任务输出

这一节我们讨论“输出模式”时，说的不是：

* RNN层输出了多少个隐藏状态

而是：

* 最终任务结果在多少个时间步上产生


#### 3. 什么叫“输出模式”？📦

##### 3.1 本质定义

所谓 RNN 的输出模式，就是在讨论：

> 输入序列有多少个时间步，最终输出结果有多少个时间步。

也就是说，它描述的是：

> 输入和输出在时间维度上的对应关系。

##### 3.2 为什么要专门讨论这个问题？

因为 RNN 和普通前馈网络不一样。

普通网络通常是：

* 输入一组特征
* 输出一个结果

但 RNN 处理的是序列，所以一个模型可能会遇到很多不同情况：

* 输入一个时间步，输出一个时间步
* 输入多个时间步，只输出一个结果
* 输入多个时间步，每一步都输出结果
* 输入一个序列，再生成另一个长度不同的序列

所以我们必须单独总结这些模式。

#### 4. RNN 输出模式的核心判断标准 🧭

##### 4.1 要判断一个任务属于哪种输出模式，只需要问自己两个问题

* 输入有几个时间步？  
  * 一个时间步 $\rightarrow 1$
  * 多个时间步 $\rightarrow N$
* 输出有几个时间步？  
  * 一个时间步 $\rightarrow 1$
  * 多个时间步 $\rightarrow N$

##### 4.2 因此最经典的模式就有四类

* $1:1$
* $1:N$
* $N:1$
* $N:N$

这里左边表示输入时间步数，右边表示输出时间步数。

所以：

* $1:1$ = 一个时间步输入，一个时间步输出
* $1:N$ = 一个时间步输入，多个时间步输出
* $N:1$ = 多个时间步输入，一个时间步输出
* $N:N$ = 多个时间步输入，多个时间步输出

#### 5. $1:1$ 模式 🔹

##### 5.1 基本含义

输入一个时间步，输出一个时间步：

$x \rightarrow RNN \rightarrow h \rightarrow FC \rightarrow y$

也就是说：

* RNN只运行一步
* 得到一个隐藏状态
* 再接输出层得到一个结果

##### 5.2 这种模式下的结构特点

这一模式中：

* 没有真正的时间展开
* 没有多步递推
* 没有历史信息积累

所以虽然结构上可以写成 RNN，但它其实并没有真正发挥“循环”特性。

##### 5.3 为什么不是 RNN 的典型使用方式

因为 RNN 最核心的价值在于：

* 处理多个时间步
* 建模前后依赖
* 利用上下文信息

而 $1:1$ 模式里根本没有“前后关系”。

所以从功能上说，它和普通神经网络差别不大。

##### 5.4 典型例子

* 输入一个时间点的传感器数据，输出一个判断值
* 输入一个单独信号，输出一个类别

这类任务通常其实没必要专门用 RNN。

#### 6. $N:1$ 模式（最常见）🔹

##### 6.1 基本含义

多个时间步输入，最后只输出一个结果：

$x_1, x_2, ..., x_N \rightarrow RNN（逐步处理）\rightarrow 取\ h_N \rightarrow FC \rightarrow y$

也就是说：

* RNN先把整个序列读完
* 得到每一步隐藏状态
* 最后只取一个隐藏状态来输出结果

##### 6.2 为什么通常取最后一个隐藏状态？

因为最后一个隐藏状态 $h_N$ 已经融合了前面整个序列的信息。

你可以把它理解成：

* 前面的时间步都在“读取信息”
* 隐藏状态不断累计上下文
* 最后一个时间步的隐藏状态，就是对整条序列的总结

所以在这种模式中，最常见的做法就是：

> 取最后一步隐藏状态，送入全连接层，输出最终结果。

##### 6.3 对应的完整模型结构

结构可以写成：

$x_1, x_2, \dots, x_N \rightarrow RNN \rightarrow h_N \rightarrow FC \rightarrow y$

如果写成 PyTorch 的思路，就是：

* RNN先处理整个序列
* 取最后时刻隐藏状态
* 再送入全连接层

##### 6.4 经典应用场景

* 文本分类：情感分析、垃圾邮件分类、新闻分类（输入一整段文本 $\rightarrow$ 输出一个类别）
* 时间序列整体预测：输入前 $10$ 天数据 $\rightarrow$ 输出第 $11$ 天的趋势类别
* 语音分类：输入一段语音信号 $\rightarrow$ 输出说话人类别 / 情感类别


#### 7. $1:N$ 模式 🔸

##### 7.1 基本含义

输入一个时间步，输出多个时间步：

$x \rightarrow RNN \rightarrow h_1, h_2, ..., h_N \rightarrow FC（每步）\rightarrow y_1, y_2, ..., y_N$

也就是说：

* 给模型一个起始输入
* 然后模型不断往后生成一个输出序列

##### 7.2 这种模式的核心特点

它的核心不是“边读边输出”，而是：

> 从一个初始信息出发，逐步展开成一个输出序列。

所以这类任务里，模型通常会：

* 先接收一个初始输入或初始状态
* 然后不断生成后续隐藏状态
* 每一步隐藏状态都映射成当前输出

##### 7.3 为什么它看起来像“生成”？

因为它不是把完整输入序列一口气读完再给结果，而是：

> 从一个起点开始，一步一步生成后续内容。

所以 $1:N$ 模式最经典的关键词就是：

* 生成
* 展开
* 逐步输出

##### 7.4 典型应用场景

* 图像描述（早期思路）  
  输入一张图像的整体特征向量，输出一句文字描述。
* 文本生成  
  输入一个主题词或一个起始 token，生成一段文本。
* 字符生成 / 序列生成  
  输入开始标记，连续生成后续字符或单词。

#### 8. 同步型 $N:N$ 模式 📚

##### 8.1 基本含义

每输入一个时间步，就对应输出一个时间步，输入和输出长度相同：

$x_1, x_2, ..., x_N \rightarrow RNN \rightarrow h_1, h_2, ..., h_N \rightarrow FC（每步）\rightarrow y_1, y_2, ..., y_N$

也就是说：

* RNN先按顺序处理整个输入序列
* 每个时间步都会产生隐藏状态
* 每个隐藏状态都接一个输出层
* 所以每一步都有对应输出

##### 8.2 这种模式的核心特点

它最大的特点是：

> 输入和输出在时间上是一一对应的。

也就是说：

* 输入第 $1$ 步，对应输出第 $1$ 步
* 输入第 $2$ 步，对应输出第 $2$ 步
* $\dots$
* 输入第 $N$ 步，对应输出第 $N$ 步

##### 8.3 这时全连接层怎么接？

在这种模式下，不是只取最后一个隐藏状态，而是：

> 每一个时间步的隐藏状态 $h_t$ 都接一个同样的全连接层。

也就是说：

$o_t = W_y h_t + b_y$

$y_t = g(o_t)$

这里输出层参数通常也是共享的。

##### 8.4 典型应用场景

* 词性标注（POS Tagging）  
  输入一句话中的每个词，输出每个词对应的词性标签。
* 命名实体识别（NER）  
  输入句子中每个词，输出每个词对应的实体标签。
* 序列中的逐步判断  
  例如每个时间步都输出当前状态判断。


#### 9. 编码器-解码器型 $N:N$ 模式 📖

##### 9.1 基本含义

先读完整个输入序列（编码），再生成另一个输出序列（解码）：

$x_1, \dots, x_N \rightarrow Encoder\ RNN \rightarrow 上下文向量\ c \rightarrow Decoder\ RNN \rightarrow y_1, \dots, y_M$

输入序列长度 $N$ 和输出序列长度 $M$ 不一定相同。

##### 9.2 为什么它和同步型 $N:N$ 不一样？

同步型 $N:N$ 的特点是：

* 输入一步，对应输出一步
* 输入输出长度相同

而编码器-解码器型 $N:N$ 的特点是：

* 先把整个输入序列读完
* 再根据整体信息逐步生成输出序列

所以它更像：

> 先理解整段输入，再重新组织成另一段输出。

##### 9.3 完整结构分成两个阶段

* Encoder 阶段  
  RNN 先读完整个输入序列，把信息压缩进最后状态或上下文表示。
* Decoder 阶段  
  另一个 RNN 以上下文表示为起点，逐步生成输出序列，每一步都接全连接层得到输出。

##### 9.4 典型应用场景

* 机器翻译  
  输入英文句子，输出中文句子。
* 文本摘要  
  输入长文本，输出较短摘要。
* 对话生成（早期思路）  
  输入一句话，输出一句回复。

#### 10. 四种模式下完整模型结构对比 📊

##### 10.1 各模式总结

* $1:1$：RNN跑一步 $\rightarrow h_1 \rightarrow FC \rightarrow y$（不体现 RNN 优势）
* $N:1$：RNN跑完整序列 $\rightarrow$ 取最后 $h_N \rightarrow FC \rightarrow y$（文本分类、情感分析）
* $1:N$：RNN从初始输入出发 $\rightarrow$ 每步 $h_t \rightarrow FC \rightarrow y_t$（图像描述、文本生成）
* $N:N$（同步）：RNN跑完整序列 $\rightarrow$ 每步 $h_t \rightarrow FC \rightarrow y_t$（序列标注、NER）
* $N:N$（编解码）：Encoder压缩 $\rightarrow$ 上下文向量 $\rightarrow$ Decoder逐步生成 $\rightarrow$ 每步接FC（机器翻译）

##### 10.2 核心规律

> 无论哪种模式，RNN层负责提取时序特征，FC层负责把隐藏状态映射成最终输出；两者缺一不可。